# BGE Reranker v2 M3 на ruMTEB

Cross-encoder `BAAI/bge-reranker-v2-m3` оценивает релевантность пары «запрос, документ». Для оценки используются готовые кандидаты из задач reranking. Основной прогон — RuBQ, отдельно можно включить русскую часть MIRACL.

[Карточка модели](https://huggingface.co/BAAI/bge-reranker-v2-m3) · [ruMTEB в MTEB](https://docs.mteb.org/overview/available_benchmarks/)

## Окружение

В Kaggle включи GPU и Internet. Локально ноутбук использует CUDA при наличии и запускается на CPU в FP32. Установка пакетов и загрузка весов требуют доступа к сети.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "mteb[transformers-v4]==2.27.22",
])

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import mteb
import torch
from sentence_transformers import CrossEncoder

In [ ]:
MODEL_ID = "BAAI/bge-reranker-v2-m3"
MAX_LENGTH = 512
RUN_MIRACL_RU = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
BATCH_SIZE = 16 if DEVICE == "cuda" else 4

kaggle_working = Path("/kaggle/working")
OUTPUT_DIR = kaggle_working if kaggle_working.is_dir() else Path.cwd()
MODEL_REVISION = mteb.get_model_meta(MODEL_ID).revision

print(f"{DEVICE=}, {DTYPE=}, {BATCH_SIZE=}, {MAX_LENGTH=}")
print(f"Ревизия модели: {MODEL_REVISION}")

## Загрузка CrossEncoder

Лимит 512 токенов применяется ко всей паре. FP16 на CUDA снижает расход VRAM и может немного изменить порядок близких оценок. Для сравнения прогонов фиксируй precision, лимит токенов и ревизии.

In [ ]:
reranker = CrossEncoder(
    MODEL_ID,
    revision=MODEL_REVISION,
    device=DEVICE,
    max_length=MAX_LENGTH,
    model_kwargs={"torch_dtype": DTYPE},
)
reranker.model.eval()

In [ ]:
query = "Какой город является столицей России?"
documents = [
    "Москва — столица России.",
    "Казань — столица Республики Татарстан.",
]

scores = reranker.predict([(query, doc) for doc in documents], show_progress_bar=False)
for index in scores.argsort()[::-1]:
    print(f"{scores[index]:.4f}  {documents[index]}")

## Задачи ruMTEB

RuBQReranking ранжирует русскоязычные абзацы по релевантности вопросу. MIRACL добавляет второй набор кандидатов и увеличивает время прогона. Скрипт печатает основной показатель, который определяет установленная версия MTEB, и сохраняет полный набор метрик.

In [ ]:
tasks = [mteb.get_task("RuBQReranking")]
if RUN_MIRACL_RU:
    tasks.append(mteb.get_task("MIRACLReranking", languages=["rus"]))

for task in tasks:
    print(task.metadata.name, task.metadata.main_score, task.metadata.dataset)

In [ ]:
run_info = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE,
    "dtype": str(DTYPE),
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "versions": {
        name: version(name)
        for name in ("mteb", "sentence-transformers", "transformers", "torch")
    },
    "tasks": {},
}

In [ ]:
for task in tasks:
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = perf_counter()

    result = mteb.evaluate(
        reranker,
        tasks=task,
        encode_kwargs={"batch_size": BATCH_SIZE},
        cache=None,
        co2_tracker=False,
    )
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    name = task.metadata.name
    run_info["tasks"][name] = {
        "dataset": task.metadata.dataset,
        "main_metric": task.metadata.main_score,
        "main_score": result[0].get_score(),
        "seconds": round(perf_counter() - started, 2),
        "peak_vram_mib": (
            torch.cuda.max_memory_reserved() / 2**20 if DEVICE == "cuda" else None
        ),
        "mteb_result": result[0].model_dump(mode="json"),
    }
    output = OUTPUT_DIR / "rumteb_reranker_results.json"
    output.write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
for name, metrics in run_info["tasks"].items():
    print(f"{name} | {metrics['main_metric']} = {metrics['main_score']:.4f}")
    print(f"Время: {metrics['seconds']:.2f} с; пик VRAM (MiB): {metrics['peak_vram_mib']}")

print(f"Полный отчёт: {output}")